# Mistake Detection - Baseline Reproduction
This notebook provides the environment setup, data preparation, and evaluation scripts to reproduce the baseline results for the Mistake Detection task on the CaptainCook4D dataset.

## 1. Environment Setup
Clone the repository, install dependencies, and mount Google Drive.

In [7]:
!git clone --recursive https://github.com/sapeirone/aml-2025-mistake-detection.git code

fatal: destination path 'code' already exists and is not an empty directory.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install torcheval

## 2. Data Preparation
Extract the pre-computed features (Omnivore and SlowFast) and model checkpoints from Google Drive.

In [10]:
# TODO: update the paths below with your own
drive_base_path = "/content/drive/MyDrive/AML_Project"
data_zip = f"{drive_base_path}/1s.zip"
ckpt_zip = f"{drive_base_path}/error_recognition_best.zip"

!mkdir -p code/data
!unzip -q -o "{data_zip}" -d code/data/

# Handle the nested zip structure for both Omnivore and SlowFast
import os
import shutil
import glob

# Define the backbones to extract
backbones = ["omnivore", "slowfast"]

for backbone in backbones:
    nested_zip = f"code/data/1s/video/{backbone}.zip"
    target_dir = f"code/data/video/{backbone}"

    if os.path.exists(nested_zip):
        print(f"Found nested zip for {backbone}: {nested_zip}")
        print(f"Unzipping to {target_dir}...")

        !mkdir -p {target_dir}
        !unzip -q -o "{nested_zip}" -d code/data/temp_extract

        # Move .npz files to the final destination
        for f in glob.glob("code/data/temp_extract/**/*.npz", recursive=True):
            shutil.move(f, target_dir)

        !rm -rf code/data/temp_extract
        print(f"{backbone} features setup complete.")
    elif os.path.exists(target_dir):
        print(f"{backbone} features directory already exists.")
    else:
        print(f"WARNING: Could not find zip or directory for {backbone}.")

# Cleanup the main extraction folder if it exists
if os.path.exists("code/data/1s"):
    !rm -rf code/data/1s

!mkdir -p code/checkpoints
!unzip -q -o "{ckpt_zip}" -d code/checkpoints/

Found nested zip for omnivore: code/data/1s/video/omnivore.zip
Unzipping to code/data/video/omnivore...
omnivore features setup complete.
Found nested zip for slowfast: code/data/1s/video/slowfast.zip
Unzipping to code/data/video/slowfast...
slowfast features setup complete.


## 3. Evaluation
Run the evaluation script for all baseline configurations (MLP/Transformer x Omnivore/SlowFast) on both Step and Recordings splits.

In [11]:
import glob
import os
import subprocess

configs = [
    # Omnivore Baselines
    {"variant": "MLP", "backbone": "omnivore", "split": "step", "threshold": 0.6, "pattern": "*MLP*omnivore*step*.pt"},
    {"variant": "MLP", "backbone": "omnivore", "split": "recordings", "threshold": 0.5, "pattern": "*MLP*omnivore*recordings*.pt"},
    {"variant": "Transformer", "backbone": "omnivore", "split": "step", "threshold": 0.6, "pattern": "*Transformer*omnivore*step*.pt"},
    {"variant": "Transformer", "backbone": "omnivore", "split": "recordings", "threshold": 0.5, "pattern": "*Transformer*omnivore*recordings*.pt"},

    # SlowFast Baselines (Added as per specs)
    {"variant": "MLP", "backbone": "slowfast", "split": "step", "threshold": 0.6, "pattern": "*MLP*slowfast*step*.pt"},
    {"variant": "MLP", "backbone": "slowfast", "split": "recordings", "threshold": 0.5, "pattern": "*MLP*slowfast*recordings*.pt"},
    {"variant": "Transformer", "backbone": "slowfast", "split": "step", "threshold": 0.6, "pattern": "*Transformer*slowfast*step*.pt"},
    {"variant": "Transformer", "backbone": "slowfast", "split": "recordings", "threshold": 0.5, "pattern": "*Transformer*slowfast*recordings*.pt"},
]

base_ckpt_dir = "code/checkpoints/error_recognition_best"

print("Starting Full Baseline Reproduction (Omnivore + SlowFast)...")

for conf in configs:
    variant = conf["variant"]
    backbone = conf["backbone"]
    split = conf["split"]
    threshold = conf["threshold"]
    pattern = conf["pattern"]

    # Find checkpoint
    search_path = os.path.join(base_ckpt_dir, variant, backbone, pattern)
    files = glob.glob(search_path)

    if not files:
        print(f"SKIPPING: No checkpoint for {variant} ({backbone}) on {split}.")
        continue

    relative_ckpt_path = files[0].replace("code/", "")

    print(f"\n{'='*60}\nRunning: {variant} ({backbone}) | Split: {split} | Threshold: {threshold}\n{'='*60}\n")

    cmd = ["python", "-m", "core.evaluate", "--variant", variant, "--backbone", backbone,
           "--ckpt", relative_ckpt_path, "--split", split, "--threshold", str(threshold)]

    try:
        process = subprocess.run(cmd, cwd="code", capture_output=True, text=True)
        print(process.stdout)
        if process.returncode != 0: print(f"Error:\n{process.stderr}")
    except Exception as e:
        print(f"Execution failed: {e}")

print("\nReproduction sequence complete.")

Starting Full Baseline Reproduction (Omnivore + SlowFast)...

Running: MLP (omnivore) | Split: step | Threshold: 0.6

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.4096162736939436, 'recall': 0.2989708115404083, 'f1': 0.3456549302643129, 'accuracy': 0.6831416629277163, 'auc': np.float64(0.6541560352028618), 'pr_auc': tensor(0.3187)}
test Step Level Metrics: {'precision': 0.6607142857142857, 'recall': 0.14859437751004015, 'f1': 0.24262295081967214, 'accuracy': 0.7105263157894737, 'auc': np.float64(0.7573902166041213), 'pr_auc': tensor(0.3638)}
----------------------------------------------------------------


Running: MLP (omnivore) | Split: recordings | Threshold: 0.5

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
----------------------------------------------------------------
test Sub Step Level Met